# Fine-tuning Whisper large-v3-turbo dengan LoRA

Notebook ini melatih adapter LoRA di atas `openai/whisper-large-v3-turbo` memakai potongan naskah latih tim. Hasilnya diekspor ke format CTranslate2 supaya bisa dipakai faster-whisper. Panduan lengkap ada di `finetune/README.md`.

Sebelum mulai:

1. Pilih Runtime > Change runtime type > T4 GPU.
2. Pastikan folder `feynman-finetune/segments/` di Google Drive sudah berisi folder rekaman dari semua pembaca.
3. Pilih Runtime > Run all. Izinkan akses Google Drive saat diminta.

Totalnya kira-kira 30–45 menit. Sebagian besar waktu habis untuk mengunduh model dan konversi.

## 1. Parameter

Ubah parameter hanya di sel ini. Semua nilainya ikut tersimpan di file log hasil run.

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/feynman-finetune"
BASE_MODEL = "openai/whisper-large-v3-turbo"
VAL_TOPICS = ["decision-tree", "sql-join"]  # 1 topik ML dan 1 topik Basis Data, tidak ikut dilatih

LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LORA_TARGETS = ["q_proj", "v_proj"]  # query dan value attention, di encoder dan decoder

LR = 5e-4
WARMUP_STEPS = 10
EPOCHS = 5
PATIENCE = 2  # berhenti kalau WER validasi tidak membaik selama 2 epoch berturut-turut
BATCH, ACCUM = 4, 2  # batch efektif 8. Kalau muncul "CUDA out of memory": BATCH, ACCUM = 2, 4
SEED = 0

## 2. Setup

Memasang library, memeriksa GPU, dan menghubungkan Google Drive. Versi `ctranslate2` dan `faster-whisper` disamakan dengan backend, supaya model hasil konversi bisa dibuka di laptop.

In [ ]:
!pip install -q peft jiwer ctranslate2==4.8.2 faster-whisper==1.2.1

import torch
from google.colab import drive

assert torch.cuda.is_available(), "Runtime belum GPU. Pilih Runtime > Change runtime type > T4 GPU, lalu jalankan ulang."
print("GPU:", torch.cuda.get_device_name(0))
drive.mount("/content/drive")

## 3. Data

Membaca semua `segments.csv`. Sel ini berhenti kalau masih ada status selain `ok` atau `buang`.

Tabel menunjukkan menit audio per pembicara dan topik. Sel kosong (NaN) berarti rekaman itu belum diunggah.

In [ ]:
import glob
import os

import pandas as pd

frames = []
for f in sorted(glob.glob(f"{DRIVE_DIR}/segments/*/segments.csv")):
    rec = os.path.basename(os.path.dirname(f))
    d = pd.read_csv(f, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    d["path"] = [os.path.join(os.path.dirname(f), x) for x in d["file"]]
    d["rec"], d["speaker"], d["topic"] = rec, rec.split("_")[1], rec.split("_")[2]
    frames.append(d)
assert frames, f"Tidak ada segments.csv di {DRIVE_DIR}/segments/"
df = pd.concat(frames, ignore_index=True)

pending = df[~df["status"].isin(["ok", "buang"])]
assert pending.empty, f"Status belum ok/buang di: {sorted(pending['rec'].unique())}. Selesaikan langkah cek dulu."

df = df[df["status"] == "ok"].copy()
df["minutes"] = (df["end"].astype(float) - df["start"].astype(float)) / 60
print(df.pivot_table(index="speaker", columns="topic", values="minutes", aggfunc="sum",
                     margins=True, margins_name="total").round(1))

val_df = df[df["topic"].isin(VAL_TOPICS)].reset_index(drop=True)
train_df = df[~df["topic"].isin(VAL_TOPICS)].reset_index(drop=True)
assert len(train_df) and len(val_df), "Data latih atau validasi kosong. Cek VAL_TOPICS."
print(f"\nlatih   : {len(train_df)} potongan, {train_df['minutes'].sum():.1f} menit")
print(f"validasi: {len(val_df)} potongan, {val_df['minutes'].sum():.1f} menit")

## 4. Fitur audio dan label

Audio diubah jadi log-Mel spectrogram (128 bin), teks diubah jadi token. Label pertama dicetak untuk dicek: harus diawali `<|id|><|transcribe|><|notimestamps|>` dan diakhiri `<|endoftext|>`.

In [ ]:
import wave

import numpy as np
from transformers import WhisperForConditionalGeneration, WhisperProcessor

processor = WhisperProcessor.from_pretrained(BASE_MODEL)
processor.tokenizer.set_prefix_tokens(language="indonesian", task="transcribe")
SOT = processor.tokenizer.convert_tokens_to_ids("<|startoftranscript|>")


def load_wav(path):
    with wave.open(path) as w:
        assert w.getframerate() == 16000 and w.getnchannels() == 1, f"bukan WAV 16 kHz mono: {path}"
        return np.frombuffer(w.readframes(w.getnframes()), np.int16).astype(np.float32) / 32768


def prepare(d):
    items = []
    for r in d.itertuples():
        feats = processor.feature_extractor(load_wav(r.path), sampling_rate=16000, return_tensors="pt").input_features[0]
        ids = processor.tokenizer(r.text).input_ids
        assert ids[0] == SOT
        # Model menaruh <|startoftranscript|> sendiri di depan input decoder, jadi token itu dibuang dari label.
        items.append({"features": feats.half(), "labels": ids[1:], "text": r.text})
    return items


train, val = prepare(train_df), prepare(val_df)
print(processor.tokenizer.decode(train[0]["labels"]))

## 5. Model dan LoRA

Bobot asli model dibekukan. Yang dilatih hanya adapter LoRA di proyeksi query dan value attention, di encoder dan decoder.

In [ ]:
import random

from peft import LoraConfig, get_peft_model

random.seed(SEED)
torch.manual_seed(SEED)
model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL)
model.generation_config.forced_decoder_ids = None  # bahasa dan task diberikan lewat generate()
model = get_peft_model(model, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                                         target_modules=LORA_TARGETS))
model.print_trainable_parameters()
model = model.cuda()

## 6. Training

WER validasi dihitung sebelum training (epoch 0) dan setelah tiap epoch, dengan decoding greedy. Adapter hanya disimpan kalau WER validasi membaik. Training berhenti sendiri kalau WER validasi tidak membaik selama `PATIENCE` epoch.

In [ ]:
import re

import jiwer

ADAPTER_DIR = "/content/adapter"


def norm(text):
    # Sama dengan finetune/segment.py: huruf kecil, imbuhan bertanda hubung disatukan, tanda baca jadi spasi.
    text = re.sub(r"(?<=\w)\s*-\s*(nya|in|an|kan)\b", r"\1", text.lower())
    text = re.sub(r"\b(di|ke|ter|nge|se)\s*-\s*(?=\w)", r"\1", text)
    return " ".join(re.sub(r"[^\w\s]", " ", text.replace("-", " ")).split())


def batches(items, size):
    for i in range(0, len(items), size):
        yield items[i:i + size]


def collate(batch):
    feats = torch.stack([b["features"] for b in batch]).float().cuda()
    labels = torch.full((len(batch), max(len(b["labels"]) for b in batch)), -100)
    for i, b in enumerate(batch):
        labels[i, :len(b["labels"])] = torch.tensor(b["labels"])
    return feats, labels.cuda()


@torch.no_grad()
def val_wer():
    model.eval()
    hyps = []
    for batch in batches(val, 8):
        feats, _ = collate(batch)
        with torch.autocast("cuda", dtype=torch.float16):
            ids = model.generate(input_features=feats, language="id", task="transcribe", max_new_tokens=225)
        hyps += processor.batch_decode(ids, skip_special_tokens=True)
    model.train()
    return jiwer.wer([norm(b["text"]) for b in val], [norm(h) for h in hyps])


opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda step: min(1.0, (step + 1) / WARMUP_STEPS))
scaler = torch.amp.GradScaler("cuda")

best = val_wer()
best_epoch, bad = 0, 0
history = [{"epoch": 0, "train_loss": None, "val_wer": best}]
model.save_pretrained(ADAPTER_DIR)
print(f"epoch 0 (tanpa fine-tuning): WER validasi {best:.1%}")

for epoch in range(1, EPOCHS + 1):
    random.shuffle(train)
    losses = []
    for i, batch in enumerate(batches(train, BATCH)):
        feats, labels = collate(batch)
        with torch.autocast("cuda", dtype=torch.float16):
            loss = model(input_features=feats, labels=labels).loss
        scaler.scale(loss / ACCUM).backward()
        losses.append(loss.item())
        if (i + 1) % ACCUM == 0 or (i + 1) * BATCH >= len(train):
            scaler.step(opt)
            scaler.update()
            opt.zero_grad()
            sched.step()
    wer = val_wer()
    history.append({"epoch": epoch, "train_loss": sum(losses) / len(losses), "val_wer": wer})
    print(f"epoch {epoch}: loss latih {history[-1]['train_loss']:.3f}, WER validasi {wer:.1%}")
    if wer < best:
        best, best_epoch, bad = wer, epoch, 0
        model.save_pretrained(ADAPTER_DIR)
    else:
        bad += 1
        if bad >= PATIENCE:
            print(f"WER validasi tidak membaik selama {PATIENCE} epoch, training dihentikan.")
            break

print(f"\nterbaik: epoch {best_epoch}, WER validasi {best:.1%} (sebelum fine-tuning {history[0]['val_wer']:.1%})")
if best_epoch == 0:
    print("Perhatian: fine-tuning tidak menurunkan WER validasi. Model hasil ekspor sama dengan model dasar.")

## 7. Gabung dan konversi

Adapter terbaik digabung ke bobot model, lalu dikonversi ke format CTranslate2 (float16). `compute_type` tetap diatur di `backend/config.py` saat model dimuat.

In [ ]:
import datetime
import gc

from huggingface_hub import hf_hub_download
from peft import PeftModel

RUN = "turbo-ft-" + datetime.datetime.now().strftime("%Y%m%d-%H%M")
MERGED_DIR, CT2_DIR = "/content/merged", f"/content/{RUN}"

del model, opt, scaler
gc.collect()
torch.cuda.empty_cache()

base = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL)
merged = PeftModel.from_pretrained(base, ADAPTER_DIR).merge_and_unload()
merged.half().save_pretrained(MERGED_DIR)
# Processor baru, tanpa prefix token dari langkah 4, supaya tokenizer yang diekspor sama dengan model dasar.
WhisperProcessor.from_pretrained(BASE_MODEL).save_pretrained(MERGED_DIR)
for f in ["tokenizer.json", "preprocessor_config.json"]:
    if not os.path.exists(f"{MERGED_DIR}/{f}"):
        hf_hub_download(BASE_MODEL, f, local_dir=MERGED_DIR)

!ct2-transformers-converter --model {MERGED_DIR} --output_dir {CT2_DIR} --quantization float16 --copy_files tokenizer.json preprocessor_config.json --force
assert os.path.exists(f"{CT2_DIR}/model.bin"), "Konversi gagal. Baca pesan error di atas."
print(sorted(os.listdir(CT2_DIR)))

## 8. Uji dan simpan

Model hasil konversi dimuat dengan faster-whisper untuk mentranskrip satu potongan validasi. Setelah itu model, adapter, dan log disimpan ke `feynman-finetune/models/` di Drive.

In [ ]:
import json
import shutil

import ctranslate2
import peft
import transformers
from faster_whisper import WhisperModel

fw = WhisperModel(CT2_DIR, device="cpu", compute_type="int8")
sample = val_df.iloc[0]
segs, _ = fw.transcribe(load_wav(sample["path"]), language="id", beam_size=5)
print("naskah:", sample["text"])
print("model :", " ".join(s.text.strip() for s in segs))

out = f"{DRIVE_DIR}/models"
os.makedirs(out, exist_ok=True)
shutil.make_archive(f"{out}/{RUN}", "zip", CT2_DIR)
shutil.copytree(ADAPTER_DIR, f"{out}/{RUN}-adapter")
log = {
    "run": RUN,
    "base_model": BASE_MODEL,
    "lora": {"r": LORA_R, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT, "targets": LORA_TARGETS},
    "training": {"lr": LR, "warmup_steps": WARMUP_STEPS, "max_epochs": EPOCHS, "patience": PATIENCE,
                 "batch": BATCH, "accum": ACCUM, "seed": SEED},
    "data": {
        "val_topics": VAL_TOPICS,
        "train_segments": len(train_df), "train_minutes": round(float(train_df["minutes"].sum()), 1),
        "val_segments": len(val_df), "val_minutes": round(float(val_df["minutes"].sum()), 1),
        "speakers": sorted(df["speaker"].unique().tolist()),
        "recordings": sorted(df["rec"].unique().tolist()),
    },
    "val_wer_note": "decoding greedy di transformers, normalisasi sama dengan finetune/segment.py",
    # Label setiap potongan yang dipakai, supaya data latih bisa ditelusuri dari log ini tanpa membuka Drive.
    "segments": [
        {"split": split, **{k: r[k] for k in ("rec", "file", "text", "start", "end")}}
        for split, d in (("train", train_df), ("val", val_df)) for r in d.to_dict("records")
    ],
    "history": history,
    "best_epoch": best_epoch,
    "versions": {"torch": torch.__version__, "transformers": transformers.__version__,
                 "peft": peft.__version__, "ctranslate2": ctranslate2.__version__},
    "gpu": torch.cuda.get_device_name(0),
}
with open(f"{out}/{RUN}.json", "w") as f:
    json.dump(log, f, indent=2)
print(f"Tersimpan di Drive feynman-finetune/models/: {RUN}.zip, {RUN}-adapter/, {RUN}.json")